In [1]:
import pandas as pd
df_malware = pd.read_csv("staDynVt2955Lab.csv")
df_benign = pd.read_csv("staDynBenignLab.csv")

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import pennylane as qml
from pennylane import numpy as pnp  # for autograd-compatible arrays
from pennylane.optimize import AdamOptimizer

In [3]:
df_benign = df_benign.drop(columns=['Unnamed: 0'], errors='ignore')

In [4]:
df = pd.concat([df_malware, df_benign], ignore_index=True)




In [5]:


X = df[['filesize', 'number_of_sections', 'sec_entropy_text','sec_entropy_data', 'sec_entropy_rsrc', 'number_of_imports',
       'imported_dll_freq', 'count_file_written', 'count_regkey_written',
       'count_dll_loaded']]
y = df['label']

In [6]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3550 entries, 0 to 3549
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   filesize              3550 non-null   int64
 1   number_of_sections    3550 non-null   int64
 2   sec_entropy_text      3550 non-null   int64
 3   sec_entropy_data      3550 non-null   int64
 4   sec_entropy_rsrc      3550 non-null   int64
 5   number_of_imports     3550 non-null   int64
 6   imported_dll_freq     3550 non-null   int64
 7   count_file_written    3550 non-null   int64
 8   count_regkey_written  3550 non-null   int64
 9   count_dll_loaded      3550 non-null   int64
dtypes: int64(10)
memory usage: 277.5 KB


In [7]:
le = LabelEncoder()
y_enc = le.fit_transform(y)

In [8]:
# Keep only numeric columns


# Then scale
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [9]:
X_scaled.shape

(3550, 10)

In [10]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [11]:
X_scaled.shape


(3550, 10)

In [12]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X_scaled)

n_qubits = 4
pca = PCA(n_components=n_qubits)
X_reduced = pca.fit_transform(X_imputed)


In [13]:
y_enc.shape

(3550,)

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X_reduced, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)


In [15]:
X_train = pnp.array(X_train, dtype=float)
X_test  = pnp.array(X_test, dtype=float)
y_train = pnp.array(y_train, dtype=int)
y_test  = pnp.array(y_test, dtype=int)

In [16]:
dev = qml.device("default.qubit", wires=n_qubits)

In [17]:
def feature_map(x):
    # x: length n_qubits
    for i in range(n_qubits):
        qml.RY(x[i], wires=i)
        # small entangling
    for i in range(n_qubits - 1):
        qml.CZ(wires=[i, i+1])

In [18]:
def variational_ansatz(params):
    # params shape: (n_layers, n_qubits, 3) for RX, RY, RZ
    n_layers = params.shape[0]
    for l in range(n_layers):
        for i in range(n_qubits):
            qml.RX(params[l, i, 0], wires=i)
            qml.RY(params[l, i, 1], wires=i)
            qml.RZ(params[l, i, 2], wires=i)
        # entangle neighboring qubits
        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i+1])

In [19]:
dev = qml.device("default.qubit", wires=n_qubits, shots=None)

In [20]:
@qml.qnode(dev, interface="autograd")
def circuit(x, params):
    qml.templates.AngleEmbedding(x, wires=range(n_qubits))
    variational_ansatz(params)
    return qml.expval(qml.PauliZ(0))

In [21]:
n_layers = 2
params = pnp.random.randn(n_layers, n_qubits, 3)

In [22]:
def quantum_predict_proba(X, params):
    probs = []
    for x in X:
        exp_v = circuit(x, params)  # Get expectation value from circuit
        # Convert to numeric if PennyLane returns ExpectationMP
        if not isinstance(exp_v, (float, int)):
            exp_v = exp_v.item() if hasattr(exp_v, "item") else float(np.array(exp_v))
        prob_1 = (1 - exp_v) / 2.0
        probs.append(prob_1)

    return np.array(probs)


def quantum_predict(X, params, threshold=0.5):
    probs = quantum_predict_proba(X, params)
    return (probs >= threshold)

In [23]:
n_layers = 3

In [24]:
rng = np.random.default_rng(42)
init_params = rng.normal(scale=0.1, size=(n_layers, n_qubits, 3))

params = pnp.array(init_params, requires_grad=True)

In [25]:
def square_loss(params, X, y):
    preds = []
    for x in X:
        ev = circuit(x, params)  # returns ArrayBox during autograd
        p1 = (1 - ev) / 2.0      # arithmetic works on ArrayBox
        preds.append(p1)
    preds = pnp.array(preds, dtype=float)  # convert AFTER computation
    y = pnp.array(y, dtype=float)
    return pnp.mean((preds - y) ** 2)

In [26]:
opt = AdamOptimizer(stepsize=0.05)
epochs = 100
batch_size = 16

In [27]:
params = pnp.array(params)  # ensure pnp type
num_train = len(X_train)

In [28]:
''' for epoch in range(epochs):
    perm = rng.permutation(num_train)
    for i in range(0, num_train, batch_size):
        idx = perm[i:i+batch_size]
        X_batch = X_train[idx]
        y_batch = y_train[idx]
        params = opt.step(lambda v: square_loss(v, X_batch, y_batch), params)

    if (epoch + 1) % 10 == 0 or epoch == 0:
        loss_val = square_loss(params, X_train, y_train)
        print(f"Epoch {epoch+1:03d}  Loss: {float(loss_val):.4f}") '''

' for epoch in range(epochs):\n    perm = rng.permutation(num_train)\n    for i in range(0, num_train, batch_size):\n        idx = perm[i:i+batch_size]\n        X_batch = X_train[idx]\n        y_batch = y_train[idx]\n        params = opt.step(lambda v: square_loss(v, X_batch, y_batch), params)\n\n    if (epoch + 1) % 10 == 0 or epoch == 0:\n        loss_val = square_loss(params, X_train, y_train)\n        print(f"Epoch {epoch+1:03d}  Loss: {float(loss_val):.4f}") '

In [29]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1️⃣ Make predictions on test set
y_pred = quantum_predict(X_test, params)

# 2️⃣ Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy:.4f}")

# 3️⃣ Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# 4️⃣ Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Test Accuracy: 0.2366

Classification Report:
              precision    recall  f1-score   support

           0       0.18      1.00      0.31       119
           1       1.00      0.08      0.15       591

    accuracy                           0.24       710
   macro avg       0.59      0.54      0.23       710
weighted avg       0.86      0.24      0.18       710


Confusion Matrix:
[[119   0]
 [542  49]]


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_curve

# ✅ Weighted square loss to handle imbalance
def weighted_square_loss(params, X, y):
    preds = []
    for x in X:
        ev = circuit(x, params)
        p1 = (1 - ev) / 2.0
        preds.append(p1)
    preds = pnp.array(preds, dtype=float)
    y = pnp.array(y, dtype=float)

    # Compute class weights inversely proportional to class counts
    w0 = 1 / (pnp.sum(y == 0) + 1e-8)
    w1 = 1 / (pnp.sum(y == 1) + 1e-8)
    weights = pnp.where(y == 0, w0, w1)

    return pnp.mean(weights * (preds - y) ** 2)


# ✅ Reinitialize optimizer and parameters (optional)
opt = AdamOptimizer(stepsize=0.05)
epochs = 100
batch_size = 16

rng = np.random.default_rng(42)
params = pnp.array(rng.normal(scale=0.1, size=(n_layers, n_qubits, 3)), requires_grad=True)
num_train = len(X_train)

# ✅ Training loop
for epoch in range(epochs):
    perm = rng.permutation(num_train)
    for i in range(0, num_train, batch_size):
        idx = perm[i:i+batch_size]
        X_batch = X_train[idx]
        y_batch = y_train[idx]
        params = opt.step(lambda v: weighted_square_loss(v, X_batch, y_batch), params)

    if (epoch + 1) % 10 == 0 or epoch == 0:
        loss_val = weighted_square_loss(params, X_train, y_train)
        print(f"Epoch {epoch+1:03d}  Weighted Loss: {float(loss_val):.4f}")


# ✅ Predictions (probabilities)
probs = quantum_predict_proba(X_test, params)

# ✅ Find best threshold automatically
prec, rec, thr = precision_recall_curve(y_test, probs)
f1_scores = 2 * prec * rec / (prec + rec + 1e-8)
best_thr = thr[np.argmax(f1_scores)]
print(f"\n🔍 Best threshold found: {best_thr:.3f}")

# ✅ Final predictions
y_pred = (probs >= best_thr)

# ✅ Evaluation
accuracy = accuracy_score(y_test, y_pred)
print(f"\n✅ Test Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


In [ ]:
import joblib

# Bundle parameters and useful objects into one dictionary
qml_model = {
    "params": params,       # trained quantum parameters
    "threshold": float(best_thr),  # best threshold for classification
    "scaler": scaler,       # StandardScaler
    "imputer": imputer,     # SimpleImputer
    "pca": pca              # PCA dimensionality reducer
}

joblib.dump(qml_model, "qml_model.pkl")

print("✅ Saved qml_model.pkl successfully!")
